# Community U-Net++ on the original grouped validation fold

Uses the same pretrained EfficientNet-B3 U-Net++, 10-epoch patch training, Dice checkpoint selection, and full-resolution inference as the completed community reproduction. Only the split changes to the original month-grouped fold0: 584 training photographs, 123 validation photographs, 197 validation annotator records. No photograph is shared. Native-resolution PQ is directly comparable to the original baseline (0.1709468443).

Attribution: Talha Celik, [Solar Filaments: U-Net++ Baseline](https://www.kaggle.com/code/realtalhacelik/solar-filaments-u-net-baseline), Apache-2.0. See ../community_exact/LICENSE. This is a grouped-split adaptation, not an exact reproduction of its original split.


In [ ]:
import sys, subprocess, importlib.util, os, json
from pathlib import Path
packages = {"segmentation_models_pytorch":"segmentation-models-pytorch", "albumentations":"albumentations", "cv2":"opencv-python-headless", "pycocotools":"pycocotools", "sklearn":"scikit-learn"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import torch
assert torch.cuda.is_available(), "This reproduction requires a Kaggle GPU."
RUN = Path('/kaggle/working/unetplusplus_grouped')
RUN.mkdir(exist_ok=True)
os.chdir(RUN)
(RUN/'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Import essential libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pycocotools.coco import COCO

# Define dataset paths
BASE_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train", "train_images")
TEST_IMG_DIR = os.path.join(BASE_DIR, "test", "test_images")
ANNOTATION_PATH = os.path.join(BASE_DIR, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")

# Check if directories exist and print file counts
train_img_count = len(os.listdir(TRAIN_IMG_DIR))
test_img_count = len(os.listdir(TEST_IMG_DIR))
print(f"Train images count: {train_img_count}")
print(f"Test images count: {test_img_count}")

# Initialize COCO API for instance annotations
print("\nLoading annotations...")
coco = COCO(ANNOTATION_PATH)

# Display basic information about the dataset
categories = coco.loadCats(coco.getCatIds())
category_names = [cat['name'] for cat in categories]

print(f"\nCategories: {category_names}")
print(f"Total annotated images: {len(coco.getImgIds())}")
print(f"Total annotations/filaments: {len(coco.getAnnIds())}")

In [ ]:
np.random.seed(42)


In [ ]:
# Import PyTorch and Albumentations
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Define the Custom PyTorch Dataset
class SolarFilamentDataset(Dataset):
    def __init__(self, coco_file, img_dir, transform=None):
        self.coco = COCO(coco_file)
        self.img_dir = img_dir
        self.transform = transform
        self.img_ids = self.coco.getImgIds()

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        # Load image info
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        # Read image in grayscale
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Load annotations and combine masks
        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        annotations = self.coco.loadAnns(ann_ids)

        # Create an empty mask
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        
        # Merge all filaments into a single binary mask
        for ann in annotations:
            mask = np.maximum(mask, self.coco.annToMask(ann))

        # Apply transformations if any
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
            # Mask should be float32 for loss functions 
            mask = mask.to(torch.float32)

        return image, mask

# Define a basic transform: Resize to 512x512 and convert to Tensor 
# Note: We expand dims because Albumentations expects HWC format for normalization
def get_transforms():
    return A.Compose([
        A.Resize(512, 512),
        A.Lambda(image=lambda x, **kwargs: np.expand_dims(x, axis=-1)), # Convert (H, W) to (H, W, 1)
        A.Normalize(mean=(0.5,), std=(0.5,)), # Normalize to [-1, 1] range 
        ToTensorV2()
    ])

# Suppress pycocotools print statements for cleaner output
from contextlib import redirect_stdout
import io

with redirect_stdout(io.StringIO()):
    # Instantiate the dataset
    train_dataset = SolarFilamentDataset(ANNOTATION_PATH, TRAIN_IMG_DIR, transform=get_transforms())

# Fetch the first sample to test the pipeline
sample_img, sample_mask = train_dataset[0]

print(f"Dataset length: {len(train_dataset)}")
print(f"Image tensor shape: {sample_img.shape}")
print(f"Mask tensor shape: {sample_mask.shape}")
print(f"Image tensor dtype: {sample_img.dtype}")
print(f"Mask tensor dtype: {sample_mask.dtype}")
print(f"Image min/max values: {sample_img.min().item():.2f} / {sample_img.max().item():.2f}")

In [ ]:
# Import necessary modules
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# 1. Fix the Lambda warning by using a regular function 
def expand_dims(image, **kwargs):
    return np.expand_dims(image, axis=-1)

# Define augmentations for training
def get_train_transforms():
    return A.Compose([
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5), # Randomly flip horizontally 
        A.VerticalFlip(p=0.5),   # Randomly flip vertically
        A.Lambda(image=expand_dims), # Safe serialization without lambda
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

# Define transforms for validation - no augmentation
def get_valid_transforms():
    return A.Compose([
        A.Resize(512, 512),
        A.Lambda(image=expand_dims),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

# 2. Split data into Train and Validation
# Note: We split unique image IDs to avoid data leakage
img_ids = coco.getImgIds()
# Match scripts/prepare_data.py exactly: seed42, five folds, validation fold0.
import random
from collections import defaultdict
month_groups=defaultdict(list)
for im in coco.dataset['images']:
    month=im.get('date_captured',Path(im['file_name']).stem[:6])[:7]
    month_groups[month].append(im['id'])
months=sorted(month_groups)
random.Random(42).shuffle(months)
month_folds={month:i%5 for i,month in enumerate(months)}
train_ids=[];valid_ids=[]
for im in coco.dataset['images']:
    month=im.get('date_captured',Path(im['file_name']).stem[:6])[:7]
    (valid_ids if month_folds[month]==0 else train_ids).append(im['id'])


print(f"Train set size: {len(train_ids)}")
print(f"Validation set size: {len(valid_ids)}")

# 3. Create a wrapper dataset to handle predefined IDs 
class FilteredSolarDataset(SolarFilamentDataset):
    def __init__(self, coco_file, img_dir, target_img_ids, transform=None):
        super().__init__(coco_file, img_dir, transform)
        self.img_ids = target_img_ids # Override with specific IDs

# Suppress pycocotools output 
from contextlib import redirect_stdout
import io

with redirect_stdout(io.StringIO()):
    train_ds = FilteredSolarDataset(ANNOTATION_PATH, TRAIN_IMG_DIR, train_ids, transform=get_train_transforms())
    valid_ds = FilteredSolarDataset(ANNOTATION_PATH, TRAIN_IMG_DIR, valid_ids, transform=get_valid_transforms())

# 4. Initialize DataLoaders
BATCH_SIZE = 8
NUM_WORKERS = 2 # Best for Kaggle default environment 

# Create loaders with pin_memory for faster GPU transfer
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Test the dataloader
train_batch_imgs, train_batch_masks = next(iter(train_loader))
print(f"Train batch images shape: {train_batch_imgs.shape}")
print(f"Train batch masks shape: {train_batch_masks.shape}")

In [ ]:
train_photos = {coco.imgs[i]['file_name'] for i in train_ids}
valid_photos = {coco.imgs[i]['file_name'] for i in valid_ids}
split_info = {'train_ids':train_ids, 'valid_ids':valid_ids, 'shared_photos':sorted(train_photos & valid_photos), 'unseen_validation_photos':sorted(valid_photos - train_photos)}
(RUN/'split.json').write_text(json.dumps(split_info, indent=2))
print('Validation photographs also in training:', len(train_photos & valid_photos))
print('Validation photographs absent from training:', len(valid_photos - train_photos))

assert not (train_photos & valid_photos), 'Training/validation photograph overlap'
assert len(train_photos)==584 and len(valid_photos)==123 and len(valid_ids)==197
import hashlib
actual=hashlib.sha256('\n'.join(sorted(Path(f).stem for f in valid_photos)).encode()).hexdigest()
assert actual == 'cb31a926b3f5587c9d5ee20288712c04f02a693a8e6b2e44dea108f4b034fa34', 'Validation fold differs from original baseline'


In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn


In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        # smp provides a robust DiceLoss implementation
        self.dice = smp.losses.DiceLoss(mode='binary')

    def forward(self, logits, targets):
        # Add channel dimension to targets: [Batch, H, W] -> [Batch, 1, H, W]
        targets = targets.unsqueeze(1) 
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return bce_loss + dice_loss

def calculate_dice_score(logits, targets, threshold=0.5):
    # Convert logits to probabilities
    probs = torch.sigmoid(logits)
    # Threshold to binary values
    preds = (probs > threshold).float()
    
    # Ensure shapes match [Batch, 1, H, W]
    targets = targets.unsqueeze(1)
    
    # Calculate intersection and union over spatial dimensions
    intersection = (preds * targets).sum(dim=(2, 3))
    union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    
    # Compute Dice with smoothing epsilon
    dice = (2. * intersection + 1e-6) / (union + 1e-6)
    
    # Return batch mean
    return dice.mean().item()

In [ ]:
import gc
import time
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. Clean up memory rigorously
torch.cuda.empty_cache()
gc.collect()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define Patch-Based Transforms
# We take 512x512 crops from the original 2048x2048 image to preserve native details
def get_patch_train_transforms():
    return A.Compose([
        A.RandomCrop(width=512, height=512), # Randomly crop a 512x512 patch (
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Lambda(image=expand_dims), # Expand dims for 1-channel
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

def get_patch_valid_transforms():
    # We resize validation to 512 just to evaluate the whole disk quickly
    return A.Compose([
        A.Resize(512, 512), 
        A.Lambda(image=expand_dims),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

# 3. Initialize Datasets and Loaders (
BATCH_SIZE = 8 # Safe for 512x512 crops 

patch_train_ds = FilteredSolarDataset(ANNOTATION_PATH, TRAIN_IMG_DIR, train_ids, transform=get_patch_train_transforms())
patch_valid_ds = FilteredSolarDataset(ANNOTATION_PATH, TRAIN_IMG_DIR, valid_ids, transform=get_patch_valid_transforms())

patch_train_loader = DataLoader(patch_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
patch_valid_loader = DataLoader(patch_valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 4. Initialize Model, Optimizer, and Loss 
print("Initializing Unet++ with EfficientNet-B3 backbone for Patch Training...")
patch_model = smp.UnetPlusPlus(encoder_name="efficientnet-b3", encoder_weights="imagenet", in_channels=1, classes=1).to(device)

criterion = BCEDiceLoss() 
patch_optimizer = optim.AdamW(patch_model.parameters(), lr=1e-3, weight_decay=1e-4)
patch_scheduler = optim.lr_scheduler.CosineAnnealingLR(patch_optimizer, T_max=10, eta_min=1e-6)

# Initialize AMP Scaler
scaler = torch.amp.GradScaler('cuda')

# 5. Training Loop
EPOCHS = 10
best_patch_val_dice = 0.0
best_patch_model_path = "best_solar_unetplusplus_patch.pth"

print(f"\nStarting Patch-Based AMP training for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    
    # --- TRAINING ---
    patch_model.train()
    train_loss, train_dice = 0.0, 0.0
    
    for images, masks in patch_train_loader:
        images, masks = images.to(device), masks.to(device)
        patch_optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = patch_model(images)
            loss = criterion(outputs, masks)
            
        scaler.scale(loss).backward()
        scaler.step(patch_optimizer)
        scaler.update()
        
        train_loss += loss.item() * images.size(0)
        train_dice += calculate_dice_score(outputs, masks) * images.size(0)
        
    train_loss /= len(patch_train_loader.dataset)
    train_dice /= len(patch_train_loader.dataset)
    
    # --- VALIDATION ---
    patch_model.eval()
    val_loss, val_dice = 0.0, 0.0
    with torch.no_grad():
        for images, masks in patch_valid_loader:
            images, masks = images.to(device), masks.to(device)
            with torch.amp.autocast('cuda'):
                outputs = patch_model(images)
                loss = criterion(outputs, masks)
                
            val_loss += loss.item() * images.size(0)
            val_dice += calculate_dice_score(outputs, masks) * images.size(0)
            
    val_loss /= len(patch_valid_loader.dataset)
    val_dice /= len(patch_valid_loader.dataset)
    patch_scheduler.step()
    
    if val_dice > best_patch_val_dice:
        best_patch_val_dice = val_dice
        torch.save(patch_model.state_dict(), best_patch_model_path)
        save_msg = "⭐ Best model saved!"
    else:
        save_msg = ""
        
    epoch_time = time.time() - epoch_start_time
    print(f"Epoch [{epoch}/{EPOCHS}] | Time: {epoch_time:.1f}s")
    print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Dice:   {val_dice:.4f}  {save_msg}")
    print("-" * 60)

print(f"\nPatch Training completed! Best Validation Dice: {best_patch_val_dice:.4f}")

In [ ]:
# Additional evaluation; does not affect source checkpoint selection or training.
from pycocotools import mask as mask_util
import csv
patch_model.load_state_dict(torch.load(best_patch_model_path))
patch_model.eval()
evaluation=[]
validation_masks=[]
for filename in sorted(valid_photos):
    original_image = cv2.imread(os.path.join(TRAIN_IMG_DIR,filename),cv2.IMREAD_GRAYSCALE)
    norm_image = (original_image / 255.0 - 0.5) / 0.5
    tensor_img = torch.tensor(norm_image,dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        probs=torch.sigmoid(patch_model(tensor_img)).cpu().numpy()[0,0]
    num_labels, labels=cv2.connectedComponents((probs>0.5).astype(np.uint8))
    pred=[mask_util.encode(np.asfortranarray(labels==i,dtype=np.uint8)) for i in range(1,num_labels) if (labels==i).sum()>=150]
    for index,rle in enumerate(pred,1):
        validation_masks.append({'filament_id':f'{Path(filename).stem}_{index}', 'segmentation_rle':rle['counts'].decode('ascii')})
    for image_id in valid_ids:
        if coco.imgs[image_id]['file_name']!=filename: continue
        truth=[coco.annToRLE(a) for a in coco.loadAnns(coco.getAnnIds(imgIds=[image_id]))]
        matrix=mask_util.iou(pred,truth,[0]*len(truth)) if pred and truth else np.zeros((len(pred),len(truth)))
        used_p=set();used_g=set();total=0.
        for value,i,j in sorted([(float(matrix[i,j]),i,j) for i,j in zip(*np.where(matrix>.5))],reverse=True):
            if i not in used_p and j not in used_g:
                used_p.add(i);used_g.add(j);total+=value
        denom=.5*(len(pred)+len(truth))
        evaluation.append({'file_name':filename,'image_id':image_id,'photo_seen_in_training':filename in train_photos,'pq':total/denom if denom else 1.0,'tp':len(used_p),'fp':len(pred)-len(used_p),'fn':len(truth)-len(used_g)})
    print('Validated',filename,flush=True)
(RUN/'validation_records.json').write_text(json.dumps(evaluation,indent=2))
unseen=[r['pq'] for r in evaluation if not r['photo_seen_in_training']]
report={'all_source_validation_pq':float(np.mean([r['pq'] for r in evaluation])), 'unseen_photo_pq':float(np.mean(unseen)) if unseen else None,'unseen_records':len(unseen),'note':'Original grouped validation fold0; no shared training photographs.', 'baseline_pq':0.1709468442532191, 'delta_vs_baseline':float(np.mean(unseen))-0.1709468442532191}
(RUN/'validation_summary.json').write_text(json.dumps(report,indent=2))
print(report)

with (RUN/'validation.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(evaluation[0]));writer.writeheader();writer.writerows(evaluation)
with (RUN/'validation_preview.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=['filament_id','segmentation_rle']);writer.writeheader();writer.writerows(validation_masks)


In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import pycocotools.mask as mask_util
from tqdm import tqdm

print("Fully Convolutional Full-Resolution Inference...")

# Load the best patch-based model 
# It was trained on 512x512, but we will feed it 2048x2048 directly!
patch_model.load_state_dict(torch.load("best_solar_unetplusplus_patch.pth"))
patch_model.eval()

AREA_THRESHOLD = 150 # Back to our proven optimal filter
submission_data_full = []

test_images = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.jpeg') or f.endswith('.jpg')]

for img_name in tqdm(test_images, desc="Full-Res Inference (Tam Çözünürlüklü Çıkarım)"):
    img_path = os.path.join(TEST_IMG_DIR, img_name)
    original_image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    # Normalize the ENTIRE 2048x2048 image at once 
    norm_image = (original_image / 255.0 - 0.5) / 0.5
    
    # Shape becomes [1, 1, 2048, 2048] 
    tensor_img = torch.tensor(norm_image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    
    with torch.no_grad(), torch.amp.autocast('cuda'):
        # Pass the full image directly through the model 
        logits = patch_model(tensor_img)
        probs = torch.sigmoid(logits).cpu().numpy()[0, 0]
        
    # Threshold at 0.5 to keep delicate structures 
    binary_mask = (probs > 0.5).astype(np.uint8)
    
    # Extract connected components 
    num_labels, labels = cv2.connectedComponents(binary_mask)
    img_id = img_name.split('.')[0]
    
    for label_idx in range(1, num_labels):
        single_filament_mask = (labels == label_idx).astype(np.uint8)
        
        # Apply optimal area filter 
        if single_filament_mask.sum() >= AREA_THRESHOLD:
            fortran_mask = np.asfortranarray(single_filament_mask)
            rle = mask_util.encode(fortran_mask)
            
            submission_data_full.append({
                "filament_id": f"{img_id}_{label_idx}",
                "segmentation_rle": rle['counts'].decode('utf-8')
            })

# Save the submission
submission_full_df = pd.DataFrame(submission_data_full)
submission_full_df.to_csv("submission_fullres.csv", index=False)

print(f"\nFull-Res Submission generated! Total filaments: {len(submission_full_df)}")
print("File 'submission_fullres.csv' is ready!)")

In [ ]:
print('Completed community reproduction:', RUN)
print('Test CSV:', RUN/'submission_fullres.csv')
print('Validation:', RUN/'validation_summary.json')

import shutil
shutil.copyfile(RUN/'submission_fullres.csv',RUN/'submission.csv')
print('Submission CSV:',RUN/'submission.csv')
print('Validation CSV:',RUN/'validation.csv')
print('Validation masks:',RUN/'validation_preview.csv')
